In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
item_meta = pd.read_csv("item_meta.csv")

print(train.shape)
print(test.shape)
print(item_meta.shape)

(147267, 3)
(15460, 3)
(7603, 15)


In [ ]:
train.head()

,item_id,user_id,timestamp
0,7926,22119,1020909887000
1,6719,3664,1072719074000
2,6719,5466,1090094915000
3,6719,16280,1152498722000
4,9677,6383,1161911421000


In [ ]:
!pip install recbole -q

In [ ]:
print("train:", train.shape)
print("test:", test.shape)
print("item_meta:", item_meta.shape)

train: (147267, 3)
test: (15460, 3)
item_meta: (7603, 15)


In [ ]:
recbole_df = train[["user_id", "item_id"]].copy()

recbole_df.columns = [
    "user_id:token",
    "item_id:token"
]

recbole_df.to_csv(
    "amazon.inter",
    sep="\t",
    index=False
)

print(recbole_df.head())

   user_id:token  item_id:token
0          22119           7926
1           3664           6719
2           5466           6719
3          16280           6719
4           6383           9677


In [ ]:
!head amazon.inter

user_id:token	item_id:token
22119	7926
3664	6719
5466	6719
16280	6719
6383	9677
7797	8421
2573	6687
8212	2169
1409	7503


In [ ]:
import os
import shutil

os.makedirs("dataset/amazon", exist_ok=True)

shutil.copy(
    "amazon.inter",
    "dataset/amazon/amazon.inter"
)

print("done")

done


In [ ]:
config = """
USER_ID_FIELD: user_id
ITEM_ID_FIELD: item_id
load_col:
    inter: [user_id, item_id]

model: LightGCN
dataset: amazon

epochs: 30
train_batch_size: 2048
eval_batch_size: 4096

embedding_size: 64
n_layers: 2

learning_rate: 0.001
reg_weight: 1e-05

eval_args:
    split: {'RS': [0.8, 0.1, 0.1]}
    group_by: user
    order: RO
    mode: full

metrics: ['Recall', 'NDCG']
topk: [10]
valid_metric: Recall@10

device: cuda
"""

with open("lightgcn.yaml", "w") as f:
    f.write(config)

print("config saved")

config saved


In [ ]:
import pandas as pd
import numpy as np
import torch
from collections import defaultdict

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

all_users = sorted(train["user_id"].unique())
all_items = sorted(train["item_id"].unique())

user2idx = {u: i for i, u in enumerate(all_users)}
item2idx = {i: j for j, i in enumerate(all_items)}

idx2user = {i: u for u, i in user2idx.items()}
idx2item = {j: i for i, j in item2idx.items()}

train["user_idx"] = train["user_id"].map(user2idx)
train["item_idx"] = train["item_id"].map(item2idx)

num_users = len(user2idx)
num_items = len(item2idx)

print(num_users, num_items)
print(train.head())

23284 13441
   item_id  user_id      timestamp  user_idx  item_idx
0     7926    22119  1020909887000     22118      7925
1     6719     3664  1072719074000      3663      6718
2     6719     5466  1090094915000      5465      6718
3     6719    16280  1152498722000     16279      6718
4     9677     6383  1161911421000      6382      9676


In [ ]:
user_items = train.groupby("user_idx")["item_idx"].apply(set).to_dict()

print(len(user_items))
print(list(user_items.items())[:1])

23284
[(0, {5657, 10923, 6460})]


In [ ]:
print("num_users:", num_users)
print("num_items:", num_items)

num_users: 23284
num_items: 13441


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

user_tensor = torch.tensor(train["user_idx"].values, dtype=torch.long)
item_tensor = torch.tensor(train["item_idx"].values, dtype=torch.long)

item_tensor_shifted = item_tensor + num_users

edge_index = torch.stack([
    torch.cat([user_tensor, item_tensor_shifted]),
    torch.cat([item_tensor_shifted, user_tensor])
])

num_nodes = num_users + num_items

deg = torch.bincount(edge_index[0], minlength=num_nodes).float()
deg_inv_sqrt = torch.pow(deg, -0.5)
deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0

values = deg_inv_sqrt[edge_index[0]] * deg_inv_sqrt[edge_index[1]]

adj = torch.sparse_coo_tensor(
    edge_index,
    values,
    (num_nodes, num_nodes)
).coalesce().to(device)

print("graph ready")
print("nodes:", num_nodes)
print("edges:", edge_index.shape[1])

/tmp/ipykernel_15984/1418643310.py:21: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj = torch.sparse_coo_tensor(


graph ready
nodes: 36725
edges: 294534


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import random
from tqdm import tqdm

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, emb_size=64, n_layers=2):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.emb_size = emb_size
        self.n_layers = n_layers

        self.user_emb = nn.Embedding(num_users, emb_size)
        self.item_emb = nn.Embedding(num_items, emb_size)

        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, adj):
        all_emb = torch.cat([
            self.user_emb.weight,
            self.item_emb.weight
        ], dim=0)

        embs = [all_emb]

        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(adj, all_emb)
            embs.append(all_emb)

        final_emb = torch.mean(torch.stack(embs, dim=1), dim=1)

        user_final = final_emb[:self.num_users]
        item_final = final_emb[self.num_users:]

        return user_final, item_final


def bpr_loss(user_e, pos_e, neg_e):
    pos_scores = torch.sum(user_e * pos_e, dim=1)
    neg_scores = torch.sum(user_e * neg_e, dim=1)
    loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
    return loss

In [ ]:
model = LightGCN(
    num_users=num_users,
    num_items=num_items,
    emb_size=64,
    n_layers=2
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

all_item_indices = np.arange(num_items)

train_pairs = train[["user_idx", "item_idx"]].values

batch_size = 2048
epochs = 20

for epoch in range(epochs):
    np.random.shuffle(train_pairs)
    total_loss = 0

    for start in tqdm(range(0, len(train_pairs), batch_size)):
        batch = train_pairs[start:start + batch_size]

        users = batch[:, 0]
        pos_items = batch[:, 1]

        neg_items = []

        for u in users:
            neg = np.random.randint(num_items)
            while neg in user_items[u]:
                neg = np.random.randint(num_items)
            neg_items.append(neg)

        users = torch.tensor(users, dtype=torch.long).to(device)
        pos_items = torch.tensor(pos_items, dtype=torch.long).to(device)
        neg_items = torch.tensor(neg_items, dtype=torch.long).to(device)

        user_final, item_final = model(adj)

        user_e = user_final[users]
        pos_e = item_final[pos_items]
        neg_e = item_final[neg_items]

        loss = bpr_loss(user_e, pos_e, neg_e)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("epoch:", epoch + 1, "loss:", total_loss)

100%|██████████| 72/72 [00:03<00:00, 18.76it/s]


epoch: 1 loss: 49.28348159790039


100%|██████████| 72/72 [00:01<00:00, 50.49it/s]


epoch: 2 loss: 49.32596307992935


100%|██████████| 72/72 [00:01<00:00, 50.53it/s]


...

epoch: 18 loss: 31.663167357444763


100%|██████████| 72/72 [00:01<00:00, 49.72it/s]


epoch: 19 loss: 31.56880831718445


100%|██████████| 72/72 [00:01<00:00, 51.28it/s]

epoch: 20 loss: 31.534708321094513


In [ ]:
model.eval()

with torch.no_grad():
    user_final, item_final = model(adj)

print(user_final.shape)
print(item_final.shape)

torch.Size([23284, 64])
torch.Size([13441, 64])


In [ ]:
test_users = sample["user_id"].unique()

test_user_idx = [
    user2idx[u]
    for u in test_users
    if u in user2idx
]

print(len(test_user_idx))

2255


In [ ]:
user_final_cpu = user_final.cpu()
item_final_cpu = item_final.cpu()

recommendations = {}

for uid in test_users:

    if uid not in user2idx:
        continue

    uidx = user2idx[uid]

    scores = torch.matmul(
        item_final_cpu,
        user_final_cpu[uidx]
    )

    interacted = user_items[uidx]

    scores[list(interacted)] = -1e9

    top_items = torch.topk(
        scores,
        k=10
    ).indices.numpy()

    recommendations[uid] = [
        idx2item[i]
        for i in top_items
    ]

print("done")

done


In [ ]:
submission = sample.copy()

submission["item_id"] = submission["user_id"].apply(
    lambda x: ",".join(
        map(str, recommendations[x])
    )
)

submission.head()

,ID,user_id,item_id
0,12,12,"2140,8421,5303,5999,4798,10326,2218,8562,3950,..."
1,14,14,"6971,2140,6220,5303,8922,9799,1290,2218,7637,9668"
2,17,17,"6060,10352,5027,5760,1377,8540,10974,4621,6952..."
3,21,21,"2140,8421,4798,2218,5999,6971,10326,6220,4413,..."
4,44,44,"1290,7120,5135,789,4929,10663,9962,2916,6243,3697"


In [ ]:
submission.to_csv(
    "submission_lightgcn.csv",
    index=False
)

print("saved")

saved


In [ ]:
popular_items = train["item_id"].value_counts().index.tolist()
popular_rank = {item: rank for rank, item in enumerate(popular_items)}

popular_score = {}

for item in popular_items:
    popular_score[item] = 1 / (popular_rank[item] + 1)

In [ ]:
popular_items_all = (
    pd.concat([train, test])["item_id"]
    .value_counts()
    .index
    .tolist()
)

In [ ]:
alpha = 0.7

recommendations_mix = {}

for uid in test_users:

    uidx = user2idx[uid]

    scores = torch.matmul(
        item_final_cpu,
        user_final_cpu[uidx]
    ).numpy()

    interacted = user_items[uidx]

    candidate_scores = {}

    top_lgcn = np.argpartition(-scores, 200)[:200]

    for item_idx in top_lgcn:
        if item_idx in interacted:
            continue

        real_item = idx2item[item_idx]
        candidate_scores[real_item] = alpha * float(scores[item_idx])

    for rank, real_item in enumerate(popular_items_all[:200]):
        item_idx = item2idx.get(real_item)

        if item_idx is None:
            continue

        if item_idx in interacted:
            continue

        candidate_scores[real_item] = candidate_scores.get(real_item, 0) + (1 - alpha) * (1 / (rank + 1))

    top10 = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]

    recommendations_mix[uid] = [item for item, score in top10]

In [ ]:
submission_mix = sample.copy()

submission_mix["item_id"] = submission_mix["user_id"].apply(
    lambda x: ",".join(map(str, recommendations_mix[x]))
)

submission_mix.to_csv("submission_lightgcn_pop_mix.csv", index=False)

submission_mix.head()

,ID,user_id,item_id
0,12,12,"2140,8421,5303,5999,4798,10326,2218,8562,3950,..."
1,14,14,"6971,1290,2140,5303,6220,8922,9799,2218,7637,9668"
2,17,17,"6060,10352,5027,5760,1377,8540,10974,4621,6952..."
3,21,21,"2140,8421,6971,2218,4798,5999,10326,6220,4413,..."
4,44,44,"1290,7120,5135,4929,789,10663,9962,2916,6243,3697"


In [ ]:
all_history = pd.concat([
    train[["user_id", "item_id", "timestamp"]],
    test[["user_id", "item_id", "timestamp"]]
])

all_history = all_history.sort_values("timestamp")

user_all_items_real = (
    all_history.groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

In [ ]:
model.eval()

with torch.no_grad():
    user_final, item_final = model(adj)

item_final_cpu = item_final.cpu()

In [ ]:
recommendations_recent = {}

for uid in sample["user_id"].unique():

    history_real = user_all_items_real.get(uid, [])

    history_idx = [
        item2idx[i]
        for i in history_real
        if i in item2idx
    ]

    if len(history_idx) == 0:
        uidx = user2idx[uid]
        user_vec = user_final.cpu()[uidx]
    else:
        recent_idx = history_idx[-5:]
        user_vec = item_final_cpu[recent_idx].mean(dim=0)

    scores = torch.matmul(
        item_final_cpu,
        user_vec
    ).numpy()

    seen_idx = set(history_idx)

    for i in seen_idx:
        scores[i] = -1e9

    top_items = np.argpartition(-scores, 50)[:50]
    top_items = top_items[np.argsort(-scores[top_items])]

    recs = [
        idx2item[i]
        for i in top_items[:10]
    ]

    recommendations_recent[uid] = recs

In [ ]:
submission_recent = sample.copy()

submission_recent["item_id"] = submission_recent["user_id"].apply(
    lambda x: ",".join(map(str, recommendations_recent[x]))
)

submission_recent.to_csv(
    "submission_recent_lightgcn.csv",
    index=False
)

submission_recent.head()

,ID,user_id,item_id
0,12,12,"2140,8421,5303,5999,4798,2218,10326,8562,3950,..."
1,14,14,"6971,2140,5303,6220,8922,9799,2218,7637,8235,1..."
2,17,17,"6060,10352,5027,5760,1377,8540,10974,4621,6306..."
3,21,21,"2140,8421,6971,2218,4798,5999,6220,10326,9799,..."
4,44,44,"1290,7120,5135,789,4929,9962,10663,2916,3697,1..."


In [ ]:
from collections import defaultdict
import math
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

all_history = pd.concat([
    train[["user_id", "item_id", "timestamp"]],
    test[["user_id", "item_id", "timestamp"]]
])

all_history = all_history.sort_values(["user_id", "timestamp"])

user_seq = all_history.groupby("user_id")["item_id"].apply(list).to_dict()

item_score = defaultdict(lambda: defaultdict(float))

for user, items in user_seq.items():
    items = list(dict.fromkeys(items))

    for i in range(len(items)):
        for j in range(i + 1, min(i + 6, len(items))):
            item_i = items[i]
            item_j = items[j]

            weight = 1 / (j - i)

            item_score[item_i][item_j] += weight
            item_score[item_j][item_i] += weight * 0.7

print("co-visitation ready")

co-visitation ready


In [ ]:
popular_items = train["item_id"].value_counts().index.tolist()

In [ ]:
def recommend_covisit(uid, topk=10):
    history = user_seq.get(uid, [])
    seen = set(history)

    scores = defaultdict(float)

    recent_items = history[-3:]

    for pos, item in enumerate(reversed(recent_items)):
        recent_weight = 1 / (pos + 1)

        for related_item, score in item_score.get(item, {}).items():
            if related_item in seen:
                continue

            scores[related_item] += score * recent_weight

    recs = [
        item for item, score in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    for item in popular_items:
        if item not in seen and item not in recs:
            recs.append(item)
        if len(recs) >= topk:
            break

    return recs[:topk]

In [ ]:
lgcn_rank = {}

for uid in sample["user_id"].unique():
    uidx = user2idx[uid]

    scores = torch.matmul(
        item_final_cpu,
        user_final_cpu[uidx]
    ).numpy()

    interacted = user_items[uidx]
    scores[list(interacted)] = -1e9

    top_items = np.argpartition(-scores, 200)[:200]
    top_items = top_items[np.argsort(-scores[top_items])]

    lgcn_rank[uid] = [idx2item[i] for i in top_items]

In [ ]:
def recommend_covisit_long(uid, topk=200):
    history = user_seq.get(uid, [])
    seen = set(history)

    scores = defaultdict(float)

    recent_items = history[-5:]

    for pos, item in enumerate(reversed(recent_items)):
        recent_weight = 1 / (pos + 1)

        for related_item, score in item_score.get(item, {}).items():
            if related_item in seen:
                continue

            scores[related_item] += score * recent_weight

    recs = [
        item for item, score in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    for item in popular_items:
        if item not in seen and item not in recs:
            recs.append(item)
        if len(recs) >= topk:
            break

    return recs[:topk]


covisit_rank = {}

for uid in sample["user_id"].unique():
    covisit_rank[uid] = recommend_covisit_long(uid, topk=200)

In [ ]:
def fuse_recommend(uid, w_cov=1.0, w_lgcn=0.7, w_pop=0.3, topk=10):
    scores = defaultdict(float)

    for rank, item in enumerate(covisit_rank[uid]):
        scores[item] += w_cov / (rank + 1)

    for rank, item in enumerate(lgcn_rank[uid]):
        scores[item] += w_lgcn / (rank + 1)

    for rank, item in enumerate(popular_items_all[:200]):
        scores[item] += w_pop / (rank + 1)

    seen = set(user_seq.get(uid, []))

    recs = [
        item for item, score in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
        if item not in seen
    ]

    return recs[:topk]

In [ ]:
import numpy as np

all_hist = pd.concat([train, test]).copy()

max_time = all_hist["timestamp"].max()

all_hist["days_ago"] = (
    max_time - all_hist["timestamp"]
) / (1000 * 60 * 60 * 24)

all_hist["time_weight"] = np.exp(
    -all_hist["days_ago"] / 30
)

time_pop = (
    all_hist.groupby("item_id")["time_weight"]
    .sum()
    .sort_values(ascending=False)
)

time_pop_items = time_pop.index.tolist()

print(time_pop_items[:10])

[9101, 854, 7679, 9730, 12470, 10326, 2411, 4798, 9942, 8413]


In [ ]:
def fuse_recommend_timepop(
    uid,
    w_cov=1.0,
    w_lgcn=0.6,
    w_pop=0.5,
    topk=10
):
    scores = defaultdict(float)

    for rank, item in enumerate(covisit_rank[uid]):
        scores[item] += w_cov / (rank + 1)

    for rank, item in enumerate(lgcn_rank[uid]):
        scores[item] += w_lgcn / (rank + 1)

    for rank, item in enumerate(time_pop_items[:200]):
        scores[item] += w_pop / (rank + 1)

    seen = set(user_seq.get(uid, []))

    recs = [
        item
        for item, score in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
        if item not in seen
    ]

    return recs[:topk]

In [ ]:
submission_timepop = sample.copy()

submission_timepop["item_id"] = (
    submission_timepop["user_id"]
    .apply(
        lambda x: ",".join(
            map(
                str,
                fuse_recommend_timepop(x)
            )
        )
    )
)

submission_timepop.to_csv(
    "submission_fusion_timepop.csv",
    index=False
)

submission_timepop.head()

,ID,user_id,item_id
0,12,12,"5303,2140,2265,9101,6824,8421,1751,854,11320,1..."
1,14,14,"7637,6971,11733,9101,12663,9668,2140,854,6220,..."
2,17,17,"5790,6060,1221,9101,7480,10352,4399,854,5027,33"
3,21,21,"7039,2140,11159,9101,2191,8421,4798,8289,854,2617"
4,44,44,"7134,1290,9101,3704,8536,7120,5135,12395,854,1..."
